# Auto Tagging for Service Now - create sample shots

### Import Data

In [ ]:
#!pip install openai

In [ ]:
# LLM Tagging with Azure OpenAI - ServiceNow Data
# Notebook Template

import pandas as pd
import numpy as np
from openai import AzureOpenAI
from sklearn.model_selection import train_test_split
import json
from collections import Counter
from typing import List, Tuple, Dict
from tqdm import tqdm
import time
import re
from dotenv import load_dotenv
import os

# =============================================================================
# 1. SETUP AND CONFIGURATION
# =============================================================================


# Select relevant columns
COLUMNS_TO_KEEP = ['Number', 'Description', 'Tags']  # Add other columns you need

# Azure OpenAI Configuration
load_dotenv()
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")  
API_KEY = os.getenv("API_KEY")
API_VERSION = "2025-01-01-preview"
DEPLOYMENT_NAME = "gpt-5"

HIERARCHICAL_CATEGORIES = [('1a MV (Mobile voice),', '2a Main product'),
('1a MV (Mobile voice),', '2a Smartwatch'),
('1a MV (Mobile voice),', '2a Datasharingcard'),
('1a MV (Mobile voice),', '2a Nummerportering'),
('1b MBB', '2b Main product'),
('1b MBB', '2b Datasharingcard'),
('1b MBB', '2b Router'),
('1b MBB', '2b Smart-sim'),
('1c TV', '2c COAX'),
('1c TV', '2c Fiber'),
('1c TV', '2c DSL'),
('1c TV', '2c OTT'),
('1d Broadband', '2d COAX'),
('1d Broadband', '2d Fiber'),
('1d Broadband', '2d DSL'),
('1e VoIP', '2e COAX'),
('1e VoIP', '2e Fiber'),
('1e VoIP', '2e DSL'),
('1e VoIP', '2e Fastnet på mobil'),
('1f PSTN', '2f Main product'),
('1f PSTN', '2f Add-on'),
('1g Self service', '2g Mit YouSee'),
('1g Self service', '2g YouSee Music'),
('1g Self service', '2g Mit Internet'),
('1g Self service', '2g YS-play'),
('1g Self service', '2g Webmail'),
('1 Misc incidents', '2 Other')
]

SCENARIOS = [
    "3 App",
    "3 Barring",
    "3 Billing/Invoices",
    "3 Change ownership",
    "3 Click & Collect",
    "3 CPR issues in Dawn",
    "3 Credit check",
    "3 CSRD",
    "3 Fraud",
    "3 Login issues",
    "3 Missing rights to access",
    "3 Mix-tv",
    "3 product status",
    "3 relocate",
    "3 onsite technician",
    "3 line technician",
    "3 Port in",
    "3 Port out",
    "3 payment method",
    "3 technical issues",
    "3 Order activation",
    "3 Order confirmation",
    "3 Performance issues",
    "3 Power of attorney",
    "3 Price and campaigns",
    "3 Proof of purchase",
    "3 Quote error",
    "3 Return/replace",
    "3 Shipment",
    "3 Sikkerhedspakke",
    "3 Stock",
    "3 Termination",
    "3 Third party",
    "3 Tickets",
    "3 User error",
    "3 Web"
]

# Initialize Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=API_KEY,
    api_version=API_VERSION
)

def test_azure_openai_connection():
    """Test Azure OpenAI connection and list deployments"""
    try:
        # Test basic connection
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=[{"role": "user", "content": "Hello"}],
        )
        print(" Connection successful!")
        return True
    except Exception as e:
        print(f" Connection failed: {e}")
        return False

def list_available_models():
    """List available models/deployments"""
    try:
        # This might not work with all API versions
        models = client.models.list()
        print("Available models:")
        for model in models.data:
            print(f"  - {model.id}")
    except Exception as e:
        print(f"Could not list models: {e}")
        print("Check your deployments in Azure portal manually")

# Test your connection
test_azure_openai_connection()
list_available_models()

In [ ]:
# =============================================================================
# 2. LLM TAGGING FUNCTION
# =============================================================================

def predict_tags_with_llm(description, examples_text, max_retries=2):
    """Predict tags using few-shot learning with Azure OpenAI"""
        
    prompt = f"""
    You are an expert in Nuuday telecommunication service classfication, top notch at categorizing Service Now for different brands such as YouSee, Telmore, Eesy, and Hiper customer tickets with super high accuracy.
    Your task is to categorize service descriptions into the correct hierarchical category.
    
    Available categories: {HIERARCHICAL_CATEGORIES}
    available scenarios: {SCENARIOS}
    
    Rules:
    1. Return EXACTLY the format: (Category, Subcategory), Scenarios
    2. Provide the top 3 most appropriate combinations of category and subcategory and scenarios based on the description 
    3. If you're unsure or if the description doesn't fit any category, return: (1 Misc incidents, 2 Other), Scenarios
    4. Be consistent and accurate in your classifications
    5. Explain your reasoning for each combination of the classification briefly.
    6. Additionally provide a confidence score for each of the 3 combinations (a score between 0 and 1). The score should represent how sure you are about the classification



    Examples:
    {examples_text}

    Input Description: {description}
    Output Tag:"""
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
            )
            predicted_tag = response.choices[0].message.content.strip()
            return predicted_tag
            
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(1)

                # clean the description and try again
                description = clean_description_with_llm(description)
            else:
                return "UNKNOWN"

In [ ]:
# =============================================================================
# 3. BATCH PREDICTION
# =============================================================================

def predict_tags_batch(df, few_shots_examples, batch_size=10):
    """Predict tags for a batch of descriptions"""
    
    predictions = []
    
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]
        batch_predictions = []
        
        print(f"Processing batch {i//batch_size + 1}/{(len(df)-1)//batch_size + 1}")


        for idx, row in batch.iterrows():
            predicted_tag = predict_tags_with_llm(
                row['Description'], 
                few_shots_examples
            )
            batch_predictions.append(predicted_tag)
            time.sleep(0.5)  # Rate limiting
        
        predictions.extend(batch_predictions)
    
    return predictions

In [ ]:
# =============================================================================
# 4. READ FEW SHOT LEARNING
# =============================================================================
few_shot_examples = """Description: The customer was supposed to have their internet and TV activated today, but the order has not gone through yet. It still shows as "Activation in progress."
Tag: (1d Broadband, 2d Fiber), 3 product status


Description: Wifi Health is Excellent despite high dBm, is this correct? Expected Result:The estimated quality of the wifi should represent the high dBm  
Tag: (1d Broadband, 2d Fiber), 3 Performance issues


Description: I get an error message that I have to use the app in Denmark before I can use it abroad. ExpectedBehaviour:Login to use yousee play without any problem.StepsToReproduce:Playback error on Yousee play  
Tag: (1g Self Service, 2g Yousee TV Play), 3 Login issues


Description: The router does not have VoIP configuration. The router has been reset.  Expected Result:  
The router should receive VoIP configuration when customers have VoIP subscriptions.  
Tag: (1e VoIP, 2e COAX), 3 technical issues


Description: The customer is experiencing problems logging into Disney+. After entering the email address and pressing "continue" the error "an error occurred" appears on the screen.  
I have informed the customer that the app is integrated and that I will let our backend know. He does not have a smartTV and therefore we cannot test if the SmartTV app has the same issue, but I have told him that he should create a free hotmail and change his Disney+ account to that, so he can get it to work - but issue should be fixed for platform and for future customers.  
Tag: (1c TV, 2c Fiber), 3 Login issues


Description: Cannot proceed with the order, Fiber should have been activated on the 16th, but the basket and order are at a standstill.
Tag: (1d Broadband, 2d Fiber, 3 Order activation"


Description: Hiper has requested the SBBU rebooking, but Dawn is not allowing it  
Tag: (1d Broadband, 2d COAX), 3 Missing rights to access"""

In [ ]:
# =============================================================================
# 5. RUN PREDICTIONS
# =============================================================================

# Running predictions
print("Starting tag predictions...")
prediction_df = pd.read_excel('untagged/24-Nov.xlsx')
prediction_df1 = prediction_df[["Number", "Description"]].copy()

# # Clean descriptions (uncomment to run - this will take time!)
# print("Cleaning descriptions with LLM...")
# prediction_df1['Description'] = prediction_df['Description'].apply(
#      lambda x: clean_description_with_llm(str(x))
# )
print(prediction_df1.head())
predicted_tags = predict_tags_batch(prediction_df1, few_shot_examples)

# Add predictions to dataframe
prediction_df1['predicted_tags'] = predicted_tags


In [ ]:
result_df = prediction_df1.copy()
result_df['Tag 1'] = result_df['predicted_tags'].str.replace(r'[()]', '', regex=True).str.split(',')
result_df['Tag 1'] = result_df['Tag 1'].apply(lambda x: [tag.strip() for tag in x])
result_df = result_df.explode('Tag 1').reset_index(drop=True)

In [ ]:
print(result_df)

In [ ]:
result_df.to_excel('24-Nov-2025-result.xlsx', index=False)